# Closed-form RNNs for discrete $SE(2)$

A vector $x\in\mathbb{R}^{|G|}$ is equivalently a scalar function $x:G\to\mathbb{R}$. This notebook uses that identification to construct a quadratic RNN whose state update realizes the left action of

$$G=\mathbb{Z}_n^2\rtimes C_3.$$

The headline is precise: **using every irrep gives an exact finite-group computation; retaining only selected Fourier blocks gives a cheaper approximation.** We first verify exactness on a small group, then run a moderate spatial experiment with power-based truncation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.discrete_se2_geometry import (
    decode_spatial_argmax,
    gaussian_bump,
    periodic_distance_squared,
    plot_group_signal,
    plot_lattice_scalar,
    spatial_marginal,
    transformed_center,
)
from src.finite_group_rnn import (
    build_finite_group_rnn,
    hidden_width,
    minimum_fourier_singular_value,
    probe_hidden_states,
    random_invertible_encoding,
    rollout,
)
from src.groups import DiscreteSE2Group

np.set_printoptions(precision=3, suppress=True)

## 1. Group and action conventions

Write $g=(t,r)$ with $t\in\mathbb{Z}_n^2$ and $r\in C_3$. For

$$A=\begin{pmatrix}-1&-1\\1&0\end{pmatrix}\pmod n,$$

the semidirect-product law is

$$(t_1,r_1)(t_2,r_2)=(t_1+A^{r_1}t_2,\ r_1+r_2).$$

Signals use the left action

$$(g\cdot x)[h]=x[g^{-1}h].$$

Consequently, relative drives $g_1,\ldots,g_T$ act in the order $g_T\cdots g_1$. The initial allocentric state is always supplied explicitly as `x_allo`; tokens never mix absolute and relative meanings.

In [ ]:
G_small = DiscreteSE2Group(n=2, m=3)
irreps_small = G_small.irreps()
dims_small = np.array([rho.dim for rho in irreps_small])

print(f"|G| = {G_small.order}")
print("irrep dimensions:", dims_small.tolist())
print("Peter–Weyl sum Σ dρ² =", int(np.sum(dims_small**2)))
assert np.sum(dims_small**2) == G_small.order


## 2. Fourier encodings and the closed-form RNN

For each irrep $\rho$,

$$\widehat{x}(\rho)=\sum_{g\in G}x(g)\rho(g)^\dagger,$$

and Peter–Weyl completeness makes the all-irrep transform lossless. We use a generic random `x_allo`; `x_ego` is sampled until every Fourier block is invertible.

With $\phi(z)=\operatorname{ReLU}(z)^2$ and relative drive $g_t$,

$$h_1=\phi(W_{\rm in}x_{\rm allo}+W_{\rm drive}(g_1\cdot x_{\rm ego})),$$
$$h_t=\phi(W_{\rm mix}h_{t-1}+W_{\rm drive}(g_t\cdot x_{\rm ego})),\qquad y_t=W_{\rm out}h_t.$$

Each irrep contributes $4q_\rho d_\rho^3=12d_\rho^3$ units for $q_\rho=3$. `apply_mix(h)` evaluates $W_{\rm in}(W_{\rm out}h)$ without materializing `W_mix`.

In [ ]:
rng = np.random.default_rng(7)
x_allo_small = rng.normal(size=G_small.order)
x_ego_small = random_invertible_encoding(G_small, irreps_small, seed=11)

x_roundtrip = G_small.inverse_fourier(G_small.fourier(x_allo_small)).real
print("Fourier round-trip max error:", np.max(np.abs(x_roundtrip - x_allo_small)))
print("minimum x_ego block singular value:",
      minimum_fourier_singular_value(x_ego_small, irreps_small))

params_small = build_finite_group_rnn(
    G_small,
    x_ego_small,
    irrep_selection="all",
    q_rho=3,
    materialize_mix=False,
)
expected_width = sum(hidden_width(rho) for rho in irreps_small)
print("hidden width:", params_small.hidden_dim)
print("W_mix materialized:", params_small.W_mix is not None)
assert params_small.hidden_dim == expected_width

## 3. Exactness checks

Because the small model includes every irrep, its decoded output should equal the regular left action. We test one-step translation and rotation, then a multistep mixed sequence. The latter is genuinely noncommutative, so it also checks multiplication order. Errors near machine precision substantiate exactness rather than merely illustrating it.

In [ ]:
exact_sequences = {
    "one-step translation": [G_small.encode(1, 0, 0)],
    "one-step rotation": [G_small.encode(0, 0, 1)],
    "mixed noncommuting": [
        G_small.encode(1, 0, 0),
        G_small.encode(0, 0, 1),
        G_small.encode(0, 1, 0),
    ],
}

for name, sequence in exact_sequences.items():
    result = rollout(params_small, x_allo_small, sequence)
    step_errors = np.max(
        np.abs(result["predicted_outputs"] - result["true_outputs"]), axis=1
    )
    print(f"{name:24s} max error = {step_errors.max():.3e}")
    assert step_errors.max() < 1e-10

translation = G_small.encode(1, 0, 0)
rotation = G_small.encode(0, 0, 1)
print("translation ∘ rotation == rotation ∘ translation:",
      G_small.compose(translation, rotation) == G_small.compose(rotation, translation))

In [ ]:
# 4. Moderate spatial experiment: deliberately truncated
# n=8 keeps the default execution practical. Four high-power irreps retain
# the dominant Fourier content while making this an approximation, not an
# exact reconstruction.
G = DiscreteSE2Group(n=8, m=3)
center_xy = (2, 3)
x_allo = gaussian_bump(G, center=center_xy, sigma=1.15)
irreps = G.irreps()
x_ego = random_invertible_encoding(G, irreps, seed=19)

params = build_finite_group_rnn(
    G,
    x_ego,
    x_allo=x_allo,
    irrep_selection="power",
    num_irreps=4,
    q_rho=3,
    materialize_mix=False,
)

print(f"|G|={G.order}, all irreps={len(irreps)}")
print("selected irrep indices:", params.selected_irrep_indices)
print("selected dimensions:", [rho.dim for rho in params.irreps])
print("hidden width:", params.hidden_dim)
print("W_mix materialized:", params.W_mix is not None)

### Interpreting the truncated experiment

Here $n=8$ gives $|G|=192$. Retaining four high-power irreps keeps the default run practical while demonstrating the approximation regime. Increase `n` and `num_irreps` gradually for higher spatial or spectral resolution; both weight storage and runtime grow, and each three-dimensional irrep contributes $12\cdot3^3=324$ hidden units.

The following cells visualize the allocentric encoding and evaluate translation-only, rotation-only, and mixed relative-drive sequences. Full signal error and decoded-center error are reported separately because truncation can preserve position while changing signal shape or amplitude.

In [ ]:
def summarize_rollout(name, sequence):
    result = rollout(params, x_allo, sequence)
    full_errors = np.linalg.norm(
        result["predicted_outputs"] - result["true_outputs"], axis=1
    ) / np.linalg.norm(result["true_outputs"], axis=1)

    center_errors = []
    for predicted, state in zip(
        result["predicted_outputs"], result["cumulative_states"]
    ):
        predicted_center = decode_spatial_argmax(G, predicted)
        exact_center = transformed_center(G, int(state), center_xy)
        center_errors.append(
            np.sqrt(periodic_distance_squared(G.n, predicted_center, exact_center))
        )

    print(
        f"{name:16s} final relative signal error={full_errors[-1]:.3e}; "
        f"max center error={max(center_errors):.3f}"
    )
    return result, np.asarray(full_errors), np.asarray(center_errors)

In [ ]:
plot_group_signal(
    G,
    x_allo,
    title="Allocentric triangular Gaussian (summed over orientation)",
    reduction="sum",
)
plt.show()

In [ ]:
translation_only = [G.encode(1, 0, 0)] * 6
rotation_only = [G.encode(0, 0, 1)] * 6
mixed = [
    G.encode(1, 0, 0),
    G.encode(0, 0, 1),
    G.encode(0, 1, 0),
    G.encode(1, 0, 1),
    G.encode(-1, 1, 0),
    G.encode(0, 0, 2),
]

spatial_results = {}
for name, sequence in {
    "translation-only": translation_only,
    "rotation-only": rotation_only,
    "mixed": mixed,
}.items():
    spatial_results[name] = summarize_rollout(name, sequence)

## 5. Hidden-state tuning

`probe_hidden_states` is a **static probe**: it evaluates the first-step hidden response while sweeping transformed allocentric inputs with a fixed drive. Those rows are not a trajectory. In contrast, `rollout(...)["hidden_states"]` contains genuinely recurrent states produced by a particular drive sequence. We show one representative static tuning map and compare its range with one recurrent unit.

In [ ]:
static_hidden = probe_hidden_states(params, x_allo)
representative = int(np.argmax(static_hidden.var(axis=0)))
static_map = spatial_marginal(G, static_hidden[:, representative])

plot_lattice_scalar(
    static_map,
    title=f"Static tuning of hidden unit {representative}",
)
plt.show()

recurrent_hidden = spatial_results["mixed"][0]["hidden_states"][:, representative]
print("static response range:", np.ptp(static_hidden[:, representative]))
print("recurrent mixed-sequence range:", np.ptp(recurrent_hidden))

In [ ]:
power = G.power_spectrum(x_allo)
retained_fraction = power[params.selected_irrep_indices].sum() / power.sum()
print(f"selected Fourier power fraction: {retained_fraction:.3%}")
print("This diagnostic describes truncation; it is not an exactness claim.")

## 6. Conclusions and limitations

- The all-irrep construction passed translation, rotation, and noncommuting multistep tests at floating-point precision; this substantiates exactness for the finite small group.
- Power-selected Fourier blocks reduce cost but make the spatial experiment approximate. Full reconstruction and decoded-center errors measure different aspects of that approximation.
- The Gaussian is copied across orientation slices. It carries no heading label, so an origin-centered isotropic bump cannot diagnose pure rotation; an off-origin bump reveals rotation only through its displaced center.
- Width scales as $12\sum_\rho d_\rho^3$ for $q_\rho=3$. Keeping `W_mix` factored avoids a dense hidden-by-hidden matrix but does not remove the cubic irrep-width cost.
- The same representation-theoretic recipe points toward discrete $SE(3)$, but larger irreps and more complex spatial geometry make truncation and factored operators even more important.